score = 0.471

In [ ]:
import os
from pathlib import Path
import datetime
from typing import List
from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler

import kaggle_evaluation.default_inference_server

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============ PATHS ============
from pathlib import Path

KAGGLE_DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')
LOCAL_DATA_PATH: Path = Path('./data')
LOCAL_CROPPED_PATH: Path = Path('./data/cropped')

# Prefer Kaggle input path when available, otherwise fall back to a local ./data folder.
DATA_PATH: Path = KAGGLE_DATA_PATH if KAGGLE_DATA_PATH.exists() else LOCAL_DATA_PATH

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                         # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                         # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                # Multiplier of predicted excess returns to signal 

# ============ MODEL CONFIGS (RIDGE) ============
CV: int = 10                                    # Number of cross validation folds
ALPHAS: np.ndarray = np.logspace(-4, 2, 100)    # Candidate ridge alphas


In [ ]:
@dataclass
class DatasetOutput:
    X_train : pl.DataFrame 
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: StandardScaler

@dataclass 
class RidgeParameters:
    cv: int
    alphas: np.ndarray

    def __post_init__(self):
        if self.cv < 2:
            raise ValueError("cv must be >= 2 for cross-validation")
        if self.alphas is None or len(self.alphas) == 0:
            raise ValueError("alphas must be a non-empty array")

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL


In [ ]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

ridge_params = RidgeParameters(
    cv = CV,
    alphas = ALPHAS
)


In [ ]:
def load_trainset_from_file(train_file_path: Path) -> pl.DataFrame:
    """
    Loads and preprocesses a specific training dataset file.

    Args:
        train_file_path: Path to the training CSV file
        
    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(train_file_path)
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )
    
    
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list[str]) -> DatasetOutput: 
    """
    Splits the data into features (X) and target (y), and scales the features.

    Args:
        train (pl.DataFrame): The processed training DataFrame.
        test (pl.DataFrame): The processed testing DataFrame.
        features (list[str]): List of features to used in model. 

    Returns:
        DatasetOutput: A dataclass containing the scaled feature sets, target series, and the fitted scaler.
    """
    X_train = train.drop(['date_id','target']) 
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target']) 
    y_test = test.get_column('target')
    
    scaler = StandardScaler() 
    
    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)
    
    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)
    
    
    return DatasetOutput(
        X_train = X_train,
        y_train = y_train, 
        X_test = X_test, 
        y_test = y_test,
        scaler = scaler
    )


In [ ]:
def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

In [ ]:
# 加载测试集（固定使用 data/test.csv）
test: pl.DataFrame = load_testset() 
print("测试集信息:")
print(test.head(3))
print(f"测试集形状: {test.shape}")


In [ ]:
# 获取所有 cropped 训练文件
import glob

cropped_train_files = sorted(glob.glob(str(LOCAL_CROPPED_PATH / "train*.csv")))
print(f"找到 {len(cropped_train_files)} 个裁剪训练文件:")
for f in cropped_train_files:
    print(f"  - {Path(f).name}")


## 结果对比表格

In [ ]:
from typing import Any
from SharpeRatio import score

def _to_numpy(x: Any) -> np.ndarray:
    """Robust conversion helper for polars/pandas/numpy inputs."""
    if hasattr(x, "to_numpy"):
        return x.to_numpy()
    return np.asarray(x)

# 存储所有结果
all_results = []
all_submissions = {}
all_parquet_submissions = {}  # 存储parquet生成的submission

# 读取原始测试集用于创建 solution
test_original = pl.read_csv(DATA_PATH / "test.csv")

# 从训练集中获取真实的forward_returns和risk_free_rate
train_original_for_solution = pl.read_csv(DATA_PATH / "train.csv")
solution_df = (
    test_original.select(['date_id'])
    .join(
        train_original_for_solution.select([
            'date_id',
            pl.col('market_forward_excess_returns').alias('forward_returns'),
            'risk_free_rate'
        ]),
        on='date_id',
        how='left'
    )
    .to_pandas()
)
solution_df['row_id'] = solution_df['date_id']

print("=" * 80)
print("开始循环测试不同的训练文件...")
print("=" * 80)

for train_file_path in tqdm(cropped_train_files, desc="处理训练文件"):
    file_name = Path(train_file_path).stem
    print(f"\n处理文件: {file_name}")
    
    try:
        # 1. 加载训练数据
        train = load_trainset_from_file(Path(train_file_path))
        
        # 检查是否有date_id列
        if 'date_id' not in train.columns:
            print(f"  ✗ 跳过: 文件中没有date_id列")
            continue
            
        # 2. 合并训练和测试数据
        df = join_train_test_dataframes(train, test)
        train_processed = df.filter(pl.col('date_id').is_in(train.get_column('date_id').to_list()))
        test_processed = df.filter(pl.col('date_id').is_in(test.get_column('date_id').to_list()))
        
        # 3. 获取特征列表
        FEATURES = [col for col in test_processed.columns if col not in ['date_id', 'target']]
        
        # 4. 分割数据集并标准化
        dataset = split_dataset(train=train_processed, test=test_processed, features=FEATURES)
        X_train = dataset.X_train
        y_train = dataset.y_train
        scaler_temp = dataset.scaler
        
        # 5. 将数据转换为numpy并填充NaN值为0
        X_train_np = _to_numpy(X_train)
        y_train_np = _to_numpy(y_train)
        
        # 检查并填充NaN值
        nan_count_X = np.isnan(X_train_np).sum()
        nan_count_y = np.isnan(y_train_np).sum()
        
        if nan_count_X > 0:
            print(f"  ⚠ 发现 {nan_count_X} 个NaN值在特征中，将填充为0")
            X_train_np = np.nan_to_num(X_train_np, nan=0.0)
        
        if nan_count_y > 0:
            print(f"  ⚠ 发现 {nan_count_y} 个NaN值在目标中，将跳过此文件")
            continue
        
        # 6. 训练模型
        model_cv = RidgeCV(alphas=ridge_params.alphas, cv=ridge_params.cv)
        model_cv.fit(X_train_np, y_train_np)
        
        best_alpha = float(model_cv.alpha_)
        
        model_temp = Ridge(alpha=best_alpha)
        model_temp.fit(X_train_np, y_train_np)
        
        # 7. 创建predict函数用于inference server
        def predict(test_df: pl.DataFrame) -> float:
            """预测函数，用于inference server"""
            test_df = test_df.rename({'lagged_forward_returns':'target'})
            test_df = test_df.with_columns(
                pl.exclude('date_id').cast(pl.Float64, strict=False)
            )
            
            # 选择特征
            X_test_single = test_df.select(FEATURES)
            
            # 检查缺失特征并填充
            missing_features = [f for f in FEATURES if f not in X_test_single.columns]
            if missing_features:
                for feat in missing_features:
                    X_test_single = X_test_single.with_columns(pl.lit(0.0).alias(feat))
            
            # 确保特征顺序一致
            X_test_single = X_test_single.select(FEATURES)
            X_test_np = X_test_single.to_numpy()
            
            # 填充NaN
            X_test_np = np.nan_to_num(X_test_np, nan=0.0)
            
            # 标准化和预测
            X_test_scaled = scaler_temp.transform(X_test_np)
            raw_pred = float(model_temp.predict(X_test_scaled)[0])
            
            return convert_ret_to_signal(raw_pred, ret_signal_params)
        
        # 8. 使用inference server生成submission.parquet
        print(f"  → 生成 submission.parquet...")
        inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
        
        # 删除旧的submission.parquet（如果存在）
        if Path('submission.parquet').exists():
            Path('submission.parquet').unlink()
        
        # 运行inference server生成parquet
        inference_server.run_local_gateway((str(DATA_PATH),))
        
        # 读取生成的submission.parquet
        if Path('submission.parquet').exists():
            submission_parquet = pd.read_parquet('submission.parquet')
            all_parquet_submissions[file_name] = submission_parquet.copy()
            
            print(f"  ✓ submission.parquet 已生成:")
            print(f"    - 形状: {submission_parquet.shape}")
            print(f"    - 预测统计: mean={submission_parquet['prediction'].mean():.4f}, "
                  f"std={submission_parquet['prediction'].std():.4f}")
            print(f"    - 前3行预测: {submission_parquet['prediction'].head(3).tolist()}")
            
            # 重命名保存当前文件的parquet
            parquet_filename = f'submission_{file_name}.parquet'
            submission_parquet.to_parquet(parquet_filename)
            print(f"  ✓ 已保存为: {parquet_filename}")
        else:
            print(f"  ⚠ submission.parquet 未生成")
            submission_parquet = None
        
        # 9. 生成本地预测（用于计算Sharpe Ratio）
        test_predictions = []
        for row in test_original.iter_rows(named=True):
            single_row_df = pl.DataFrame([row]).with_columns(
                pl.exclude('date_id').cast(pl.Float64, strict=False)
            ).rename({'lagged_forward_returns':'target'})
            
            X_test_single = single_row_df.select(FEATURES)
            
            missing_features = [f for f in FEATURES if f not in X_test_single.columns]
            if missing_features:
                for feat in missing_features:
                    X_test_single = X_test_single.with_columns(pl.lit(0.0).alias(feat))
            
            X_test_single = X_test_single.select(FEATURES)
            X_test_np = X_test_single.to_numpy()
            X_test_np = np.nan_to_num(X_test_np, nan=0.0)
            X_test_scaled_np = scaler_temp.transform(X_test_np)
            
            raw_pred = float(model_temp.predict(X_test_scaled_np)[0])
            signal = convert_ret_to_signal(raw_pred, ret_signal_params)
            test_predictions.append(signal)
        
        # 10. 创建 submission DataFrame
        submission_df = pd.DataFrame({
            'date_id': test_original['date_id'].to_list(),
            'row_id': test_original['date_id'].to_list(),
            'prediction': test_predictions
        })
        
        # 11. 计算 Sharpe Ratio
        sharpe_score_val = score(
            solution=solution_df, 
            submission=submission_df, 
            row_id_column_name='date_id'
        )
        
        # 12. 保存结果
        all_results.append({
            'file_name': file_name,
            'threshold': file_name.replace('train_filtered_threshold_', '').replace('p', '.'),
            'sharpe_ratio': sharpe_score_val,
            'best_alpha': best_alpha,
            'train_samples': len(train),
            'num_features': len(FEATURES),
            'nan_filled': nan_count_X,
            'parquet_generated': submission_parquet is not None,
            'mean_prediction': np.mean(test_predictions),
            'std_prediction': np.std(test_predictions),
            'min_prediction': np.min(test_predictions),
            'max_prediction': np.max(test_predictions)
        })
        
        all_submissions[file_name] = submission_df
        
        print(f"  ✓ Sharpe Ratio: {sharpe_score_val:.6f}")
        print(f"  ✓ Best Alpha: {best_alpha:.6g}")
        print(f"  ✓ 训练样本数: {len(train)}")
        print(f"  ✓ 特征数量: {len(FEATURES)}")
        
    except Exception as e:
        print(f"  ✗ 处理失败: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "=" * 80)
print("所有文件处理完成！")
print(f"成功处理: {len(all_results)} / {len(cropped_train_files)} 个文件")
print(f"Parquet生成: {sum(1 for r in all_results if r.get('parquet_generated', False))} 个")
print("=" * 80)

In [ ]:
# 展示每个模型的前几行预测
print("=" * 100)
print("各模型 Submission 预测对比（前10行）")
print("=" * 100)

comparison_display = pd.DataFrame({'date_id': test_original['date_id'].to_list()[:10]})
for file_name, submission in all_submissions.items():
    threshold = file_name.replace('train_filtered_threshold_', '').replace('p', '.')
    comparison_display[f'pred_{threshold}'] = submission['prediction'].values[:10]

print(comparison_display.to_string(index=False))
print("\n注意: 仅显示前10行，完整数据保存在各个submission中")

In [ ]:
# 对比parquet生成的预测与本地计算的预测
if len(all_parquet_submissions) > 0:
    print("=" * 100)
    print("Parquet 生成的预测对比（前10行）")
    print("=" * 100)
    
    parquet_comparison = pd.DataFrame({'date_id': test_original['date_id'].to_list()[:10]})
    for file_name, parquet_sub in all_parquet_submissions.items():
        threshold = file_name.replace('train_filtered_threshold_', '').replace('p', '.')
        parquet_comparison[f'parquet_{threshold}'] = parquet_sub['prediction'].values[:10]
    
    print(parquet_comparison.to_string(index=False))
    
    # 验证parquet和本地计算的一致性
    print("\n" + "=" * 100)
    print("验证 Parquet vs 本地计算的一致性")
    print("=" * 100)
    
    for file_name in all_parquet_submissions.keys():
        if file_name in all_submissions:
            parquet_pred = all_parquet_submissions[file_name]['prediction'].values
            local_pred = all_submissions[file_name]['prediction'].values
            
            # 计算差异
            diff = np.abs(parquet_pred - local_pred)
            max_diff = diff.max()
            mean_diff = diff.mean()
            
            print(f"\n{file_name}:")
            print(f"  - 最大差异: {max_diff:.6f}")
            print(f"  - 平均差异: {mean_diff:.6f}")
            print(f"  - 是否一致: {'✓' if max_diff < 1e-6 else '✗'}")
else:
    print("❌ 没有生成的 parquet submission")

## 对比 Parquet 生成的预测

In [ ]:
# 创建结果对比表格
if len(all_results) == 0:
    print("❌ 没有成功处理的文件，无法生成对比表格")
    print("请检查训练文件格式和列名是否正确")
else:
    results_comparison_df = pd.DataFrame(all_results)
    
    # 按 Sharpe Ratio 降序排序
    results_comparison_df = results_comparison_df.sort_values('sharpe_ratio', ascending=False)
    
    print("\n" + "=" * 100)
    print("不同训练文件的模型表现对比")
    print("=" * 100)
    print(results_comparison_df.to_string(index=False))
    print("=" * 100)
    
    # 找出最佳模型
    best_result = results_comparison_df.iloc[0]
    print(f"\n🏆 最佳模型:")
    print(f"  - 文件名: {best_result['file_name']}")
    print(f"  - 阈值: {best_result['threshold']}")
    print(f"  - Sharpe Ratio: {best_result['sharpe_ratio']:.6f}")
    print(f"  - Best Alpha: {best_result['best_alpha']:.6g}")
    print(f"  - 训练样本数: {int(best_result['train_samples'])}")
    print(f"  - 特征数量: {int(best_result['num_features'])}")
    print(f"  - 平均预测信号: {best_result['mean_prediction']:.4f}")
    
    # 保存结果表格到CSV
    results_comparison_df.to_csv('ridge_cropped_comparison.csv', index=False)
    print(f"\n结果已保存到: ridge_cropped_comparison.csv")

In [ ]:
# 显示最佳模型的 submission 内容
best_file_name = best_result['file_name']
best_submission = all_submissions[best_file_name]

print(f"最佳模型 ({best_file_name}) 的 Submission 内容:")
print("=" * 80)
print(best_submission.head(20))
print("\n...")
print(best_submission.tail(20))
print("=" * 80)

print(f"\nSubmission 统计信息:")
print(best_submission['prediction'].describe())

# 保存最佳模型的 submission
best_submission.to_csv('best_ridge_submission.csv', index=False)
print(f"\n最佳模型的 submission 已保存到: best_ridge_submission.csv")


In [ ]:
import matplotlib.pyplot as plt

# 创建图表
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Sharpe Ratio vs Threshold
ax1 = axes[0, 0]
results_comparison_df['threshold_float'] = results_comparison_df['threshold'].astype(float)
results_comparison_df_sorted = results_comparison_df.sort_values('threshold_float')
ax1.plot(results_comparison_df_sorted['threshold_float'], 
         results_comparison_df_sorted['sharpe_ratio'], 
         marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Threshold', fontsize=12)
ax1.set_ylabel('Sharpe Ratio', fontsize=12)
ax1.set_title('Sharpe Ratio vs Feature Selection Threshold', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5)

# 2. Training Samples vs Sharpe Ratio
ax2 = axes[0, 1]
ax2.scatter(results_comparison_df['train_samples'], 
           results_comparison_df['sharpe_ratio'],
           s=100, alpha=0.6, c=results_comparison_df['sharpe_ratio'], 
           cmap='viridis')
ax2.set_xlabel('Training Samples', fontsize=12)
ax2.set_ylabel('Sharpe Ratio', fontsize=12)
ax2.set_title('Training Samples vs Sharpe Ratio', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. Best Alpha vs Threshold
ax3 = axes[1, 0]
ax3.semilogy(results_comparison_df_sorted['threshold_float'], 
             results_comparison_df_sorted['best_alpha'],
             marker='s', linewidth=2, markersize=8, color='green')
ax3.set_xlabel('Threshold', fontsize=12)
ax3.set_ylabel('Best Alpha (log scale)', fontsize=12)
ax3.set_title('Ridge Alpha vs Feature Selection Threshold', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Prediction Statistics
ax4 = axes[1, 1]
x = np.arange(len(results_comparison_df_sorted))
width = 0.35
ax4.bar(x - width/2, results_comparison_df_sorted['mean_prediction'], 
        width, label='Mean', alpha=0.8)
ax4.bar(x + width/2, results_comparison_df_sorted['std_prediction'], 
        width, label='Std', alpha=0.8)
ax4.set_xlabel('Threshold', fontsize=12)
ax4.set_ylabel('Prediction Value', fontsize=12)
ax4.set_title('Prediction Statistics by Threshold', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(results_comparison_df_sorted['threshold'].values, rotation=45)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ridge_cropped_comparison.png', dpi=300, bbox_inches='tight')
print("图表已保存到: ridge_cropped_comparison.png")
plt.show()
